# Component 2 Notebook: Soft Domain Context

This notebook is an interface to test and run Component 2 (TorchXRayVision DenseNet121 routing head) without touching the source Python directly.
The core architecture lives in `src/components/component2_txv.py` and the training protocol runs via `src/training/train_component2_txv.py`.

It mirrors the Component 1 workflow by dynamically loading the `paths.yaml` config and checking your external HDD structure.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

REPO_ROOT

In [ ]:
import torch
import matplotlib.pyplot as plt
import subprocess
from pprint import pprint

from src.training.train_component1_dann import load_yaml_config, build_component1_manifest
from src.components.component2_txv import Component2SoftDomainContext
from src.training.train_component2_txv import Component2DomainDataset

## Load Config

Ensure your layout points correctly to your `checkpoints` directory and raw imaging structures.

In [ ]:
PATHS_CONFIG = REPO_ROOT / "configs" / "paths.yaml"
COMPONENT2_CONFIG = REPO_ROOT / "configs" / "component2_txv.yaml"
COMPONENT1_CONFIG = REPO_ROOT / "configs" / "component1_dann.yaml"

component2_cfg = load_yaml_config(COMPONENT2_CONFIG)["component2_txv"]

print("Model Backend:", component2_cfg["model"]["backend"])
print("Expected Epochs:", component2_cfg["training"]["epochs"])
print("Learning Rate:", component2_cfg["training"]["lr"])

## Forward Model Pass (Validation)

Validate that the shape dimensions match the exact spec tensor values:
`img_emb: [B, 256]` mapped down from `[B, 1024]`.

In [ ]:
device = torch.device("cpu")
model = Component2SoftDomainContext(
    backend=component2_cfg["model"]["backend"],
    weights=component2_cfg["model"]["weights"]
).to(device)

model.eval()

# Load a real sample from the domain manifest mapped from Component 1 paths
manifest = build_component1_manifest(paths_config=PATHS_CONFIG, component1_config=COMPONENT1_CONFIG)
dataset = Component2DomainDataset(manifest)
sample = dataset[0]

# Visualise raw
x_224 = sample["x_224"]
domain_id = int(sample["domain_id"])

plt.figure(figsize=(4, 4))
# Denormalise [-1024, 1024] purely for display mapping roughly to [0, 1]
disp_img = (x_224[0].cpu().numpy() + 1024) / 2048.0
plt.imshow(disp_img, cmap="gray")
plt.title(f"{sample['dataset_id']} | domain={domain_id}")
plt.axis("off")
plt.show()

# Forward pass on actual tensor
with torch.no_grad():
    # Model expects [B, 1, 224, 224], adding batch dimension
    domain_ctx, pathology_logits = model(x_224.unsqueeze(0))

print("Input: ", tuple(x_224.unsqueeze(0).shape))
print("Domain context routing tensor `domain_ctx`: ", tuple(domain_ctx.shape))
print("Pathology logits vector `pathology_logits`: ", tuple(pathology_logits.shape))
print("L2 Norm check of domain_ctx: ", torch.norm(domain_ctx, p=2, dim=1).item())

## Train Domain Routing Head

Run a `--dry-run` to ensure your external dataset structure maps to Component 2 logic without errors. Subprocess is used to keep global states clean.

In [ ]:
DRY_RUN = True

cmd = [
    sys.executable,
    "-m",
    "src.training.train_component2_txv",
]
if DRY_RUN:
    cmd.append("--dry-run")

result = subprocess.run(cmd, cwd=REPO_ROOT, capture_output=True, text=True)
print(result.stdout)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f"Training script failed with status code {result.returncode}")